# Assignment VOL

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_excel('/Users/jlaw/projects/stern/systematic-investing/data/assignment_VOL_data.xlsx')
pd.set_option('display.max_columns', None) # Show all columns when printing a dataframe
df.head()

## Part A: Setup Data and Indicators

### Step 1: Calculate Daily Returns and 20-day Volatility

In [ ]:
# Calculate daily returns (% change from previous day)
# Starts at row 2 (index 1) since we need 2 prices
df['daily_return'] = df['Close'].pct_change()

# Calculate 20-day rolling volatility of daily returns
# Starts at row 21 (index 20) since we need 20 daily returns
df['vol20'] = df['daily_return'].rolling(window=20).std()

# Calculate 20-day historical return (cumulative return over last 20 days using 21 prices)
# Starts at row 21 (index 20) since we need 21 prices
df['ret20'] = (1 + df['daily_return']).rolling(window=21).apply(lambda x: np.prod(x) - 1, raw=False)

# Calculate 20-day future return (cumulative return over next 20 days)
# This requires looking ahead, so we shift daily returns back by 20
df['fret20'] = (1 + df['daily_return']).shift(-20).rolling(window=21).apply(lambda x: np.prod(x) - 1, raw=False)

# Display first few rows with data
print("Data with daily returns and volatility:")
print(df[['Date', 'Close', 'daily_return', 'vol20', 'ret20', 'fret20']].iloc[20:30])

### Step 2: Calculate Z-Scores (Normalized Volatility and Returns)

Using a 250-day rolling window for mean and standard deviation

In [ ]:
# Function to calculate z-score using rolling 250-day window
# Z(i) = [X(i) - AVERAGE(X(i-1)...X(i-250))] / STDEV(X(i-1)...X(i-250))
def calculate_zscore(series, window=250):
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()
    zscore = (series - rolling_mean) / rolling_std
    return zscore

# Calculate z-scores for vol20, ret20, and fret20
# These start at row 271 (index 270) since we need 250 prior values
df['zvol20'] = calculate_zscore(df['vol20'], window=250)
df['zret20'] = calculate_zscore(df['ret20'], window=250)
df['zfret20'] = calculate_zscore(df['fret20'], window=250)

# Display where valid data starts (row 272 in Excel = index 271)
print("First valid zvol20/zret20/zfret20 data (row 272 in Excel):")
print(df[['Date', 'zvol20', 'zret20', 'zfret20']].iloc[270:280])
print(f"\nTotal rows: {len(df)}")
print(f"Valid data starts at index: 270 (row 272 in Excel)")
print(f"Valid data ends at index: {len(df)-21} (last 20 rows don't have fret20)")

### Step 3: Prepare Clean Dataset for Analysis

Remove rows with missing data and create quintile buckets

In [ ]:
import matplotlib.pyplot as plt

# Create clean dataset: remove rows with NaN in required columns
# Need zvol20, zret20, zfret20 to all have values
df_clean = df.dropna(subset=['zvol20', 'zret20', 'zfret20']).copy()

# Remove last 20 rows that don't have forward return data
df_clean = df_clean.iloc[:-20].copy()

print(f"Clean dataset size: {len(df_clean)} rows")
print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")

# Create quintile buckets based on zvol20
# pd.qcut creates equal-frequency buckets (quintiles)
df_clean['zvol20_quintile'] = pd.qcut(df_clean['zvol20'], q=5, labels=['Q1 (Low Vol)', 'Q2', 'Q3', 'Q4', 'Q5 (High Vol)'], duplicates='drop')

print("\nQuintile distribution:")
print(df_clean['zvol20_quintile'].value_counts().sort_index())
print("\nZvol20 ranges by quintile:")
for q in ['Q1 (Low Vol)', 'Q2', 'Q3', 'Q4', 'Q5 (High Vol)']:
    if q in df_clean['zvol20_quintile'].values:
        subset = df_clean[df_clean['zvol20_quintile'] == q]['zvol20']
        print(f"{q}: {subset.min():.4f} to {subset.max():.4f}")

## Part A: Analysis Questions

### Question 1: Concurrent Relationship between zvol20 and zret20

Is there a concurrent relationship between normalized volatility and normalized returns?

In [ ]:
# Group by quintile and calculate average zvol20 and zret20 for each quintile
q1_analysis = df_clean.groupby('zvol20_quintile').agg({
    'zvol20': 'mean',
    'zret20': 'mean'
}).reset_index()

print("Q1: Concurrent Relationship (zvol20 vs zret20)")
print("="*50)
print(q1_analysis)
print()

# Calculate correlation
correlation_concurrent = df_clean['zvol20'].corr(df_clean['zret20'])
print(f"Correlation between zvol20 and zret20: {correlation_concurrent:.4f}")
print()

# Plot concurrent relationship
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(q1_analysis['zvol20'], q1_analysis['zret20'], s=100, alpha=0.6)
ax.plot(q1_analysis['zvol20'], q1_analysis['zret20'], 'b-', alpha=0.3)

# Add quintile labels
for idx, row in q1_analysis.iterrows():
    ax.annotate(f"Q{idx+1}", (row['zvol20'], row['zret20']), 
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Average Normalized Volatility (zvol20)', fontsize=12)
ax.set_ylabel('Average Normalized Returns (zret20)', fontsize=12)
ax.set_title('Q1: Concurrent Relationship - Volatility vs Returns', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation: Is there a concurrent relationship between volatility and returns?")
print("(e.g., do higher volatility periods see lower/higher returns?)")

### Question 2: Lead-Lag Relationship between zvol20 and zfret20

Is there a lead-lag relationship between normalized volatility and FUTURE normalized returns?

In [ ]:
# Group by quintile and calculate average zvol20 and zfret20 for each quintile
q2_analysis = df_clean.groupby('zvol20_quintile').agg({
    'zvol20': 'mean',
    'zfret20': 'mean'
}).reset_index()

print("Q2: Lead-Lag Relationship (zvol20 vs zfret20)")
print("="*50)
print(q2_analysis)
print()

# Calculate correlation
correlation_leadlag = df_clean['zvol20'].corr(df_clean['zfret20'])
print(f"Correlation between zvol20 and zfret20: {correlation_leadlag:.4f}")
print()

# Plot lead-lag relationship
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(q2_analysis['zvol20'], q2_analysis['zfret20'], s=100, alpha=0.6, color='orange')
ax.plot(q2_analysis['zvol20'], q2_analysis['zfret20'], color='orange', alpha=0.3)

# Add quintile labels
for idx, row in q2_analysis.iterrows():
    ax.annotate(f"Q{idx+1}", (row['zvol20'], row['zfret20']), 
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Average Normalized Volatility (zvol20)', fontsize=12)
ax.set_ylabel('Average Future Normalized Returns (zfret20)', fontsize=12)
ax.set_title('Q2: Lead-Lag Relationship - Volatility vs Future Returns', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation: Does today's volatility predict future returns?")
print("(This would suggest a tradeable anomaly if relationship is strong and consistent)")

### Summary of Findings

**Relationships observed:**

In [ ]:
print("SUMMARY OF FINDINGS")
print("="*60)
print()
print("1. CONCURRENT RELATIONSHIP (zvol20 vs zret20):")
print(f"   Correlation: {correlation_concurrent:.4f}")
if abs(correlation_concurrent) < 0.1:
    print("   Interpretation: Negligible relationship between volatility and returns")
elif correlation_concurrent > 0:
    print("   Interpretation: Higher volatility associated with higher returns (positive feedback)")
else:
    print("   Interpretation: Higher volatility associated with lower returns (volatility drag)")
print()

print("2. LEAD-LAG RELATIONSHIP (zvol20 vs zfret20):")
print(f"   Correlation: {correlation_leadlag:.4f}")
if abs(correlation_leadlag) < 0.1:
    print("   Interpretation: Volatility does not predict future returns")
elif correlation_leadlag > 0:
    print("   Interpretation: High volatility predicts higher future returns")
else:
    print("   Interpretation: High volatility predicts lower future returns (mean reversion?)")
print()

print("Possible Economic Explanations:")
print("- Volatility Drag: High volatility periods may see lower returns due to risk premium")
print("- Mean Reversion: Extreme volatility may be followed by return reversals")
print("- Volatility Clustering: High volatility today predicts high volatility tomorrow,")
print("  which could affect future returns through various mechanisms")
print("- Leverage/Deleveraging: Risk-off events increase volatility and suppress returns")
print("="*60)

## Part B: Extra Credit - Trading Strategy

If the lead-lag relationship is strong, develop a trading strategy based on zvol20 predicting zfret20

In [ ]:
# Simple strategy: Trade based on zvol20 quintiles
# Long (buy) when volatility is in Q1 (low), short when in Q5 (high)
# Based on assumption that low vol predicts higher future returns

df_clean['trading_signal'] = 0
df_clean.loc[df_clean['zvol20_quintile'] == 'Q1 (Low Vol)', 'trading_signal'] = 1  # Long
df_clean.loc[df_clean['zvol20_quintile'] == 'Q5 (High Vol)', 'trading_signal'] = -1  # Short
# Neutral in Q2, Q3, Q4

# Calculate strategy returns (signal * next 20-day returns)
df_clean['strategy_returns'] = df_clean['trading_signal'] * df_clean['fret20']

# Compare strategy to buy-and-hold
df_clean['bnh_cumulative'] = (1 + df_clean['fret20']).cumprod() - 1
df_clean['strategy_cumulative'] = (1 + df_clean['strategy_returns']).cumprod() - 1

# Calculate Sharpe Ratio
# Assuming 0 risk-free rate for simplicity (or adjust with actual Rf)
Rf_annual = 0.02  # Approximate risk-free rate
Rf_20day = Rf_annual / (252/20)  # Convert to 20-day return

sharpe_bnh = (df_clean['fret20'].mean() - Rf_20day) / df_clean['fret20'].std() * np.sqrt(252/20)
sharpe_strategy = (df_clean['strategy_returns'].mean() - Rf_20day) / df_clean['strategy_returns'].std() * np.sqrt(252/20)

# Information Ratio = (Strategy Return - Benchmark Return) / Tracking Error
tracking_error = (df_clean['strategy_returns'] - df_clean['fret20']).std() * np.sqrt(252/20)
excess_return = (df_clean['strategy_returns'].mean() - df_clean['fret20'].mean()) * (252/20)
information_ratio = excess_return / tracking_error if tracking_error > 0 else 0

print("STRATEGY PERFORMANCE ANALYSIS")
print("="*60)
print()
print("Signal Distribution:")
print(df_clean['trading_signal'].value_counts())
print()

print("Cumulative Returns:")
print(f"Buy-and-Hold (S&P500):  {df_clean['bnh_cumulative'].iloc[-1]:.4f} ({df_clean['bnh_cumulative'].iloc[-1]*100:.2f}%)")
print(f"Vol-Based Strategy:     {df_clean['strategy_cumulative'].iloc[-1]:.4f} ({df_clean['strategy_cumulative'].iloc[-1]*100:.2f}%)")
print()

print("Risk-Adjusted Performance:")
print(f"Buy-and-Hold Sharpe Ratio:  {sharpe_bnh:.4f}")
print(f"Strategy Sharpe Ratio:      {sharpe_strategy:.4f}")
print()

print(f"Tracking Error:             {tracking_error:.4f}")
print(f"Excess Annual Return:       {excess_return:.4f} ({excess_return*100:.2f}%)")
print(f"Information Ratio:          {information_ratio:.4f}")
print()

print("Analysis:")
if abs(information_ratio) > 0.5:
    print(f"✓ Strong strategy signal (IR = {information_ratio:.2f})")
elif abs(information_ratio) > 0.2:
    print(f"◐ Moderate strategy signal (IR = {information_ratio:.2f})")
else:
    print(f"✗ Weak strategy signal (IR = {information_ratio:.2f})")
print("="*60)

In [ ]:
# Visualize strategy performance
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Cumulative Returns Comparison
axes[0].plot(df_clean.index, df_clean['bnh_cumulative'], label='Buy-and-Hold (S&P500)', linewidth=2)
axes[0].plot(df_clean.index, df_clean['strategy_cumulative'], label='Vol-Based Strategy', linewidth=2)
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Strategy Performance vs Buy-and-Hold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Rolling 20-day returns colored by signal
colors = df_clean['trading_signal'].map({1: 'green', -1: 'red', 0: 'gray'})
axes[1].bar(df_clean.index, df_clean['fret20'], color=colors, alpha=0.6)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Days')
axes[1].set_ylabel('20-Day Forward Return')
axes[1].set_title('Forward Returns Colored by Signal (Green=Long, Red=Short, Gray=Neutral)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()